In [1]:
import pandas as pd
import numpy as np
import requests
from bs4 import BeautifulSoup
import sqlite3
from datetime import datetime

csv_file = 'countries_by_gdp.csv'
db_name = 'world_economies.db'
table_name = 'countries_by_gdp'
log_file = 'etl_project_log.txt'

In [2]:
def extract():
 
    url = (
        "https://web.archive.org/web/20230902185326/"
        "https://en.wikipedia.org/wiki/List_of_countries_by_GDP_%28nominal%29"
    )
    df = pd.DataFrame(columns=["Pays/Territoire","Estimation"])   # Crée un dataframe vide
    
    count = 0    # Initialisation pour boucle
    
    # Récupération du tableau
    html_page = requests.get(url).text
    data = BeautifulSoup(html_page, 'html.parser')
    table = data.find("table", {"class": "wikitable"})
    
    rows = table.find_all('tr')
    
    # Extraction des données
    for row in rows:
        if count < 213:
            col = row.find_all('td')
            if len(col) >= 3:  # Vérifie qu’il y a au moins 3 colonnes
                pays = col[0].get_text(strip=True)
                estimation = col[2].get_text(strip=True).replace(',', '')
                try:
                    estimation = float(estimation)
                    df1 = pd.DataFrame({"Pays/Territoire": [pays],
                                        "Estimation": [estimation]})
                    df = pd.concat([df, df1], ignore_index=True)
                    count += 1
                except ValueError:
                    # Ignore les lignes où l’estimation n’est pas numérique
                    continue
        else:
            break
    
    return df

In [3]:
def transform(df):
    df_transformed = data_extracted.copy()    # Crée une copie des données extraites

    df_transformed.columns = ['county', 'gdp_usd_billion']   # Renomage de colonnes
    
# Convertion de million à milliard
    df_transformed['gdp_usd_billion'] = (df_transformed['gdp_usd_billion'] / 1000).round(2)
    return df_transformed 

In [4]:
# Chargement des données sous format csv
def load_data_to_csv():
    data_transformed.to_csv(csv_file, index = False)

In [5]:
# Chargement des données du fichier csv dans une base de données
def load_data_to_db():
    conn = sqlite3.connect(db_name)    # Connexion et création de la base de données dans sqlite3
    df = pd.read_csv('countries_by_gdp.csv')
    df.to_sql(table_name, conn, if_exists = "replace", index = False)
    
# Requête sql
    query_statement = f"SELECT * FROM {table_name} WHERE gdp_usd_billion >= 100"
    query_output = pd.read_sql(query_statement, conn)
    print(query_statement)
    print(query_output)
    
    conn.close()    # Fermeture de la connexion

In [6]:
#Ce script permet d'écrire des messages sur le fichier log et d'horodater
def log_progress(message): 
    timestamp_format = '%Y-%h-%d-%H:%M:%S' # Year-Monthname-Day-Hour-Minute-Second 
    now = datetime.now() # get current timestamp 
    timestamp = now.strftime(timestamp_format) 
    with open(log_file,"a") as f: 
        f.write(timestamp + ',' + message + '\n') 

In [7]:
#Dans ce script on a les différentes messages à écrire dans le fichier log pour chaque étape du processus ETL
#Début du processus ETL
log_progress("ETL Job Started") 
 
#Début de l'extraction
log_progress("Extract phase Started") 
data_extracted = extract()
print("Data extracted")
 
#Fin de l'extraction
log_progress("Extract phase Ended") 

#Début de la transformation
log_progress("Transform phase Started") 
data_transformed = transform(data_extracted)
print("Data Transformed") 
 
#Fin de la transformation
log_progress("Transform phase Ended") 

#Début du chargement
log_progress("Load phase Started") 
data_loaded = load_data_to_csv()
print("Data succesfully loaded to csv file")
loaded_data_to_db = load_data_to_db()
print("Data succesfully loaded in db")
 
#Fin du chargement
log_progress("Load phase Ended") 
 
#Fin du processus ETL
log_progress("ETL Job Ended") 

C:\Users\baaro\AppData\Local\Temp\ipykernel_10612\3842079997.py:29: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, df1], ignore_index=True)


Data extracted
Data Transformed
Data succesfully loaded to csv file
SELECT * FROM countries_by_gdp WHERE gdp_usd_billion >= 100
           county  gdp_usd_billion
0           World        105568.78
1   United States         26854.60
2           China         19373.59
3           Japan          4409.74
4         Germany          4308.85
..            ...              ...
65          Kenya           118.13
66         Angola           117.88
67           Oman           104.90
68      Guatemala           102.31
69       Bulgaria           100.64

[70 rows x 2 columns]
Data succesfully loaded in db
